<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L08-threat-model-an-agent/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L08-threat-model-an-agent/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T10-L08-threat-model-an-agent/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T10-L08-threat-model-an-agent/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T10-L08 · Threat-model an agent: a machine-readable risk register, and the tests that fail when a mitigation is missing

**You will build:** a threat model of an agent system kept as data, not as a diagram: where untrusted text
can travel, which of the ten categories of an agentic-application risk list each component is exposed to,
the checklist of mitigations that follows, and executable checks that fail until each mitigation is present.

**Time:** ~45 minutes · **Runs on:** a laptop CPU, no GPU, standard library only · **Prerequisites:**
T07-L01-agent-loop-from-scratch, T07-L02-mcp-from-the-wire-up

The agent system here is a **seeded example**: a fixed support assistant with a web fetcher, a mail sender, a
code runner, a memory, a third-party tool server and a human approver. It is invented for this lesson. The
lesson teaches a method for turning a threat model into tests; it does not say your system has these holes.

By the end you will be able to:

1. Implement a finder for the data flows that cross a trust boundary, and their direction.
2. Implement a taint propagation that finds every component untrusted text can reach, cycles included.
3. Implement a risk register that maps components and flows to the categories of the OWASP list.
4. Implement a mitigation checklist generated from the register, merging duplicates.
5. Implement a check runner that fails closed, and measure how the failures fall as mitigations are added.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L01`, `T07-L02`

**This lesson assumes you can already:**

- use Python sets and frozensets: subset tests, union, and building a set up in a loop
- read a nested dict and a list of dicts, and sort tuples of strings
- say what an LLM agent does in a loop, calling tools and reading their results (T07-L01 builds one; T07-L02 builds the tool wire)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Threat Modeling Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Threat_Modeling_Cheat_Sheet.html) · OWASP Cheat Sheet Series · *free* — the four questions of threat modelling and the data-flow-diagram habit that exercises 1 and 2 turn into code
- [The lethal trifecta for AI agents](https://simonwillison.net/2025/Jun/16/the-lethal-trifecta/) · Simon Willison · *free* — a short worked argument for why private data, untrusted content and an outward channel together are the dangerous combination, the idea behind the rules table in section 4
- [MCP specification 2025-11-25: Tools](https://modelcontextprotocol.io/specification/2025-11-25/server/tools) · Model Context Protocol project · *free* — the tool definition and its security considerations, the third-party tool server component in the seeded architecture
- [Threat Modeling: Designing for Security](https://www.wiley.com/en-us/Threat+Modeling%3A+Designing+for+Security-p-9781118809990) · Adam Shostack (Wiley) · *paid* — the standard book on trust boundaries and data flow diagrams, if you want the method behind exercise 1 in depth

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import copy
import sys
from typing import Callable

print("python", sys.version.split()[0])

# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 60     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 2048     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---

# Every check and every demo below runs through `_try`, so pressing Run all before you have written
# a line reaches the progress board at the foot of the notebook instead of stopping at the first stub.
_FAILED_CHECKS: list = []
_STATUS: dict = {}          # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}      # label -> the stub whose NotImplementedError stopped it

# The exercises, in the order the notebook meets them: (label, function, where).
_EXERCISES = [("exercise 1", "boundary_crossings", "section 2"),
              ("exercise 2", "tainted_components", "section 3"),
              ("exercise 3", "risk_register", "section 4"),
              ("exercise 4", "mitigation_checklist", "section 5"),
              ("exercise 5", "run_checks", "section 6")]
_FUNCTION = {label: function for label, function, _ in _EXERCISES}


def _unfinished(exc: BaseException) -> str:
    """The function that raised NotImplementedError: the innermost frame of the traceback."""
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (`longer_win_rate`)"; several exercises -> "exercises 3 and 4"."""
    if len(labels) > 1 and all(label in _FUNCTION for label in labels):
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} (`{_FUNCTION[label]}`)" if label in _FUNCTION else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet says so, by name. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken exercise
    never hides the feedback on the others. A demo names the exercises it `needs`: until each has
    passed its check, the demo says which one it is waiting for and skips. Nothing is swallowed:
    every outcome is recorded for the progress board, and a script run exits non-zero there if any
    check came back wrong. A check that runs longer than its limit is stopped and counts as failed,
    so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, function in _FUNCTION.items() if function == stub and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises NotImplementedError.")
        elif label in _FUNCTION:
            print(f"{label}: not implemented yet, `{stub or 'a stub'}` still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
        else:
            print(f"{label}: skipped, `{stub or 'a stub'}` is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The setting: a threat model you can run

A threat model on a whiteboard goes stale the day it is drawn. Kept as **data** it can be queried,
diffed in review and turned into tests. This lesson keeps three things as data:

* the **architecture**: components, each with a *trust level* (0 text from outside, 1 a tool or a third
  party, 2 the agent's own core and memory, 3 a person) and capability *tags*, and the **data flows**
  between them, each with tags (a flow tagged `content` carries free text that the receiver reads);
* a **rules table** that says which tags make a component or flow exposed to which category of the
  OWASP Top 10 for Agentic Applications (identifiers ASI01 to ASI10);
* a **configuration** for each item, which the checks at the end read.

The identifiers and titles below are those of the published PDF (December 2025), as of the date in `claims.yaml`; the list is new and has
been revised since its first draft. Everything else here, the rules, the mitigation catalogue and the
checks, is this course's own simplification, not OWASP's, and is a teaching device, not a standard.
The system and its settings are invented.

In [ ]:
ASI_IDS = tuple(f"ASI{n:02d}" for n in range(1, 11))
# Identifiers and titles of the OWASP Top 10 for Agentic Applications 2026, read 2026-10-20 (see claims.yaml).
ASI_TITLES = {
    "ASI01": "Agent Goal Hijack", "ASI02": "Tool Misuse and Exploitation", "ASI03": "Identity and Privilege Abuse",
    "ASI04": "Agentic Supply Chain Vulnerabilities", "ASI05": "Unexpected Code Execution (RCE)",
    "ASI06": "Memory & Context Poisoning", "ASI07": "Insecure Inter-Agent Communication",
    "ASI08": "Cascading Failures", "ASI09": "Human-Agent Trust Exploitation", "ASI10": "Rogue Agents",
}
TRUST_NAMES = {0: "outside", 1: "tool", 2: "agent", 3: "person"}


def _c(trust, *tags):
    return {"trust": trust, "tags": frozenset(tags)}


def _f(fid, src, dst, *tags):
    return {"id": fid, "src": src, "dst": dst, "tags": frozenset(tags)}


COMPONENTS = {
    "web": _c(0), "inbox": _c(0),
    "user": _c(3, "human"), "approver": _c(3, "human", "approves"),
    "orchestrator": _c(2, "llm", "agent", "holds_credentials", "delegates", "unattended"),
    "research_agent": _c(2, "llm", "agent"),
    "memory": _c(2, "persistent_memory"),
    "fetch_tool": _c(1, "tool", "reads_untrusted"),
    "mail_tool": _c(1, "tool", "sends_external", "holds_credentials"),
    "code_tool": _c(1, "tool", "executes_code"),
    "mcp_server": _c(1, "tool", "third_party"),
}
FLOWS = [
    _f("f01", "user", "orchestrator", "content"),
    _f("f02", "web", "fetch_tool", "content"),
    _f("f03", "fetch_tool", "research_agent", "content"),
    _f("f04", "research_agent", "orchestrator", "content", "agent_to_agent"),
    _f("f05", "orchestrator", "research_agent", "content", "agent_to_agent"),
    _f("f06", "orchestrator", "mail_tool", "content"),
    _f("f07", "orchestrator", "code_tool", "content"),
    _f("f08", "orchestrator", "memory", "content"),
    _f("f09", "memory", "orchestrator", "content"),
    _f("f10", "inbox", "orchestrator", "content"),
    _f("f11", "orchestrator", "mcp_server", "content"),
    _f("f12", "mcp_server", "orchestrator", "content"),
    _f("f13", "orchestrator", "approver", "content"),
    _f("f14", "approver", "orchestrator", "approval"),
]
# The settings each item runs with today. A key that is absent means the setting was never made.
CONFIG = {
    "orchestrator": {"wraps_untrusted": False, "scopes": ["*"], "token_ttl_s": 86400, "max_steps": 200,
                     "max_usd": 50, "log": "stdout", "kill_switch": False},
    "research_agent": {"wraps_untrusted": True},
    "mail_tool": {"allowed_tools": ["*"], "schema_validated": False, "approval": "optional",
                  "scopes": ["mail.*"], "token_ttl_s": 7200},
    "mcp_server": {"version": "latest", "allowed_tools": ["*"]},
    "code_tool": {"sandbox": "process", "network": "open"},
    "memory": {"validate_writes": False},
    "f05": {"auth": "none"},
    "approver": {"show_source": False},
}
ARCH = {"components": COMPONENTS, "flows": FLOWS, "config": CONFIG}


def check_architecture(components, flows):
    """Given. Raise ValueError, naming the culprit, if the architecture is not well formed."""
    seen = set()
    for f in flows:
        if f["id"] in seen or f["id"] in components:
            raise ValueError(f"flow id {f['id']!r} is used twice, or is also a component name")
        seen.add(f["id"])
        for end in ("src", "dst"):
            if f[end] not in components:
                raise ValueError(f"flow {f['id']!r} has an unknown {end} component {f[end]!r}")
    for name, c in components.items():
        if c["trust"] not in TRUST_NAMES:
            raise ValueError(f"component {name!r} has trust {c['trust']!r}; use one of {sorted(TRUST_NAMES)}")


def _describe() -> None:
    check_architecture(COMPONENTS, FLOWS)
    print(f"{len(COMPONENTS)} components, {len(FLOWS)} data flows, {sum('content' in f['tags'] for f in FLOWS)} of them carrying free text")
    for level, name in TRUST_NAMES.items():
        members = [n for n, c in COMPONENTS.items() if c["trust"] == level]
        print(f"  trust {level} ({name}): {', '.join(members)}")


if __name__ == "__main__":
    _describe()

## 2. Exercise 1 — `boundary_crossings`: where does data change trust level?

A **trust boundary** is any flow whose two ends are at different trust levels. Two directions matter for
different reasons. Data moving **up** (from a lower level to a higher one) is how untrusted text reaches
something that acts on it. Data moving **down** is how a secret or a private result leaves.

Write `boundary_crossings(components, flows)`: a dict from flow id to `"up"` (source trust below
destination trust) or `"down"` (above), containing only the flows that cross a boundary, in the order of
`flows`. Call the given `check_architecture` first so a flow that names a missing component raises
`ValueError`.

<details><summary>💡 Hint 1 — what to think about</summary>

Each flow has a source and a destination, and each of those is a name to look up in `components`.
What do you compare, and what should happen when the two are equal?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Go through the flows once, in order. Look up the trust of both ends, skip the flow when they are equal,
otherwise record the direction under the flow's id. The order of a dict is the order you insert in.

</details>

In [ ]:
def boundary_crossings(components, flows):
    """The flows that cross a trust boundary, and which way.

    Args:
        components: dict, component name -> {"trust": int, "tags": frozenset}.
        flows: list of dicts with "id", "src", "dst", "tags".

    A flow that names a component not in `components` raises ``ValueError`` (use `check_architecture`).

    Returns:
        dict flow id -> "up" or "down", only for flows whose ends differ in trust, in the order of `flows`.

    Example:
        >>> comps = {"a": {"trust": 0, "tags": frozenset()}, "b": {"trust": 2, "tags": frozenset()}}
        >>> boundary_crossings(comps, [{"id": "x", "src": "a", "dst": "b", "tags": frozenset()}])
        {'x': 'up'}
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
_TINY = {"a": {"trust": 0, "tags": frozenset()}, "b": {"trust": 2, "tags": frozenset()}, "c": {"trust": 2, "tags": frozenset()},
         "d": {"trust": 1, "tags": frozenset()}}


def _check_boundary_crossings() -> None:
    flows = [{"id": "fz", "src": "a", "dst": "b", "tags": frozenset()}, {"id": "fy", "src": "b", "dst": "c", "tags": frozenset()},
             {"id": "fm", "src": "d", "dst": "a", "tags": frozenset()}, {"id": "fb", "src": "c", "dst": "a", "tags": frozenset()},
             {"id": "fk", "src": "a", "dst": "d", "tags": frozenset()}]
    got = boundary_crossings(_TINY, flows)
    assert dict(got) == {"fz": "up", "fm": "down", "fb": "down", "fk": "up"}, (
        f"got {dict(got)}. A flow crosses a boundary only when its two ends differ in trust; 'up' is from a lower "
        "trust level to a higher one. A flow between two components at the same level is not a crossing.")
    assert list(got) == ["fz", "fm", "fb", "fk"], "keep the order of `flows` (the ids here are not in sorted order); a difference of one level counts"
    try:
        boundary_crossings(_TINY, [{"id": "q", "src": "a", "dst": "nowhere", "tags": frozenset()}])
    except ValueError:
        pass
    else:
        raise AssertionError("a flow that names an unknown component must raise ValueError")
    print("exercise 1 looks right")


_try("exercise 1", _check_boundary_crossings)

In [ ]:
def _show_crossings() -> None:
    got = boundary_crossings(COMPONENTS, FLOWS)
    ups = [fid for fid, d in got.items() if d == "up"]
    downs = [fid for fid, d in got.items() if d == "down"]
    by_id = {f["id"]: f for f in FLOWS}
    print(f"{len(got)} of {len(FLOWS)} flows cross a trust boundary: {len(ups)} up, {len(downs)} down")
    entries = [fid for fid in ups if COMPONENTS[by_id[fid]["src"]]["trust"] == 0]
    print("entry points, flows that bring text from outside (trust 0) into the system: "
          + ", ".join(f"{fid} ({by_id[fid]['src']} -> {by_id[fid]['dst']})" for fid in entries))
    inside = [fid for fid in ups if fid not in entries]
    print(f"{len(inside)} more flows move up a level inside the system, so a boundary check on the entry points alone would not see them: "
          + ", ".join(inside))


_try("boundaries", _show_crossings, needs=("exercise 1",))

## 3. Exercise 2 — `tainted_components`: how far does outside text travel?

The entry points are not the whole story. Text that enters at the fetcher is read by a research agent,
which passes a summary to the orchestrator, which writes to memory, calls tools and shows a person a
summary. Everything downstream of an untrusted source is **tainted**: not necessarily compromised, but
reachable by whoever controls that source.

Write `tainted_components(components, flows)`. The **sources** are the components at trust 0. A flow
tagged `"content"` carries its source's text to its destination; a flow without that tag (an approval
signal, say) carries none. Return the sorted list of component names that are the destination of a
content flow from a source or from an already-tainted component. A source is in the list only if some
content flow from a tainted or source component reaches it. The architecture has cycles: do not loop.

<details><summary>💡 Hint 1 — what to think about</summary>

One step from a source is not enough: what the first step reaches can pass the text on. Think of a set
that grows until a whole pass over the flows adds nothing, and of what stops that from running for ever
when two components send text to each other.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Keep a set of names that already carry outside text (start it with the sources) and a separate set of
names reached. Repeat over the content flows: when a flow's source carries text, its destination is reached
and now carries text too. Stop when a pass changes nothing. Return the reached names sorted.

</details>

In [ ]:
def tainted_components(components, flows):
    """Every component that text from a trust-0 source can reach through content flows.

    Args:
        components: dict, name -> {"trust": int, "tags": frozenset}. The sources are those with trust 0.
        flows: list of dicts with "id", "src", "dst", "tags". Only flows tagged "content" carry text.

    A flow that names an unknown component raises ``ValueError``. Cycles are normal.

    Returns:
        A sorted list of names: those that are the destination of a content flow whose source is a
        source or already tainted. A source appears only if such a flow reaches it.

    Example:
        >>> comps = {"s": {"trust": 0, "tags": frozenset()}, "a": {"trust": 2, "tags": frozenset()},
        ...          "b": {"trust": 2, "tags": frozenset()}}
        >>> fl = [{"id": "1", "src": "s", "dst": "a", "tags": frozenset({"content"})},
        ...       {"id": "2", "src": "a", "dst": "b", "tags": frozenset({"content"})}]
        >>> tainted_components(comps, fl)
        ['a', 'b']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_tainted_components() -> None:
    n = frozenset
    comps = {"s": {"trust": 0, "tags": n()}, "zz": {"trust": 2, "tags": n()}, "aa": {"trust": 2, "tags": n()},
             "c": {"trust": 2, "tags": n()}, "t": {"trust": 1, "tags": n()}, "e": {"trust": 2, "tags": n()},
             "w": {"trust": 0, "tags": n()}}
    fl = [{"id": "3", "src": "aa", "dst": "zz", "tags": n({"content"})},
          {"id": "2", "src": "zz", "dst": "aa", "tags": n({"content"})},
          {"id": "1", "src": "s", "dst": "zz", "tags": n({"content"})},
          {"id": "4", "src": "aa", "dst": "c", "tags": n({"approval"})},
          {"id": "5", "src": "t", "dst": "e", "tags": n({"content"})},
          {"id": "6", "src": "zz", "dst": "s", "tags": n({"content"})}]
    got = tainted_components(comps, fl)
    assert isinstance(got, list) and got == ["aa", "s", "zz"], (
        f"got {got}. Follow content flows as far as they go, however the flows are listed (aa to zz is listed before s to zz) and "
        "round the cycle; do not follow a flow that is not tagged content (aa to c); only trust-0 components are sources (t, at "
        "trust 1, is not); a source is listed only if a tainted component sends it text (s yes, w no); return a sorted list.")
    print("exercise 2 looks right")


_try("exercise 2", _check_tainted_components)

In [ ]:
def _one_hop(components, flows):
    """Given, for comparison: only the destinations of content flows that start at a source."""
    sources = {n for n, c in components.items() if c["trust"] == 0}
    return sorted({f["dst"] for f in flows if "content" in f["tags"] and f["src"] in sources})


def _show_taint() -> None:
    check_architecture(COMPONENTS, FLOWS)
    full, near = tainted_components(COMPONENTS, FLOWS), _one_hop(COMPONENTS, FLOWS)
    print(f"text from outside reaches {len(full)} of {len(COMPONENTS)} components: {', '.join(full)}")
    print(f"flows leaving a source reach only {len(near)}: {', '.join(near)}")
    missed = [n for n in full if n not in near]
    print(f"{len(missed)} components a one-step view misses, among them {', '.join(n for n in missed if COMPONENTS[n]['trust'] == 3) or 'no person'} (trust 3, a person)")
    untouched = [n for n in COMPONENTS if n not in full]
    print(f"not reached: {', '.join(untouched)}")


_try("taint", _show_taint, needs=("exercise 2",))

## 4. The rules, and Exercise 3 — `risk_register`

A **rule** says: *an item with all of these tags is exposed to this category; and, if the rule says so,
only when untrusted text can reach it.* A rule is `(category, scope, tags, needs_taint)`, where scope is
`"component"` or `"flow"`. A flow counts as reached when its **destination** is tainted. Read the table
in the next cell: it is data, so a team can review and change it without touching code.

In [ ]:
RULES = [
    ("ASI01", "component", frozenset({"llm"}), True),
    ("ASI02", "component", frozenset({"tool", "sends_external"}), True),
    ("ASI02", "component", frozenset({"tool", "holds_credentials"}), True),
    ("ASI02", "component", frozenset({"tool", "third_party"}), True),
    ("ASI03", "component", frozenset({"holds_credentials"}), False),
    ("ASI04", "component", frozenset({"third_party"}), False),
    ("ASI05", "component", frozenset({"executes_code"}), True),
    ("ASI06", "component", frozenset({"persistent_memory"}), True),
    ("ASI07", "flow", frozenset({"agent_to_agent"}), False),
    ("ASI08", "component", frozenset({"delegates"}), False),
    ("ASI09", "component", frozenset({"approves"}), True),
    ("ASI10", "component", frozenset({"unattended"}), False),
]
print(f"{len(RULES)} rules over {len({r[0] for r in RULES})} of the {len(ASI_IDS)} categories")

### Exercise 3

Write `risk_register(components, flows, tainted, rules)`: a dict with **one key for every category in
`ASI_IDS`**, each value the sorted list of ids of the items exposed to it (component names, or flow ids
for a flow rule). An item matches a rule when it carries **all** the rule's tags and, if the rule needs
taint, it is tainted. An item can match two rules of one category and must be listed once. A rule that
names a category not in `ASI_IDS`, or a scope other than the two, raises `ValueError`.

<details><summary>💡 Hint 1 — what to think about</summary>

Start from the ten empty categories, so a category nobody is exposed to still appears. For a rule, which
items does it look at (components or flows), what is "all the tags" in terms of sets, and what does a flow's
taint depend on? Two rules of the same category may match the same item.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Keep a set per category. For each rule, go through components or flows according to its scope, test that the
rule's tags are a subset of the item's tags, test taint when the rule needs it, and add the item's id to the
category's set. A set removes duplicates. At the end sort every set into a list.

</details>

In [ ]:
def risk_register(components, flows, tainted, rules):
    """Map every category to the items exposed to it.

    Args:
        components, flows: as in exercise 1.
        tainted: the names from exercise 2 (a flow is reached when its destination is in this collection).
        rules: list of (category, scope, tags, needs_taint).

    A rule with a category outside ``ASI_IDS`` or a scope that is not "component" or "flow" raises ``ValueError``.

    Returns:
        dict with exactly the keys of ``ASI_IDS`` in that order; each value a sorted list of item ids, no repeats.

    Example:
        >>> comps = {"m": {"trust": 2, "tags": frozenset({"persistent_memory"})}}
        >>> risk_register(comps, [], [], [("ASI06", "component", frozenset({"persistent_memory"}), False)])["ASI06"]
        ['m']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_risk_register() -> None:
    comps = {"m": {"trust": 2, "tags": frozenset({"persistent_memory", "tool"})},
             "t": {"trust": 1, "tags": frozenset({"tool", "sends_external"})},
             "u": {"trust": 1, "tags": frozenset({"tool"})}}
    fl = [{"id": "f1", "src": "u", "dst": "t", "tags": frozenset({"agent_to_agent"})}]
    rules = [("ASI06", "component", frozenset({"persistent_memory"}), True),
             ("ASI02", "component", frozenset({"tool", "sends_external"}), True),
             ("ASI02", "component", frozenset({"sends_external"}), False),
             ("ASI07", "flow", frozenset({"agent_to_agent"}), True)]
    rules = rules + [("ASI03", "component", frozenset({"tool"}), False)]
    got = risk_register(comps, fl, ["t", "u"], rules)
    assert list(got) == list(ASI_IDS), "one key for every category, in ASI_IDS order, including the empty ones"
    assert got["ASI06"] == [], "ASI06 needs taint and memory is not tainted here"
    assert got["ASI02"] == ["t"], (
        f"got {got['ASI02']}: the item matches two rules of the same category and is listed once; the tool 'u' lacks "
        "'sends_external', and a rule needs ALL of its tags")
    assert got["ASI03"] == ["m", "t", "u"], f"got {got['ASI03']}: each category's list is sorted"
    got = risk_register(comps, fl + [{"id": "f2", "src": "t", "dst": "u", "tags": frozenset({"agent_to_agent"})}], ["t"], rules)
    assert got["ASI07"] == ["f1"], "a flow is reached when its DESTINATION is tainted: f1 goes to t, f2 goes to u"
    for bad in (("ASI11", "component", frozenset(), False), ("ASI01", "edge", frozenset(), False)):
        try:
            risk_register(comps, fl, [], [bad])
        except ValueError:
            continue
        raise AssertionError("a category outside ASI_IDS, or a scope that is neither component nor flow, must raise ValueError")
    print("exercise 3 looks right")


_try("exercise 3", _check_risk_register)

In [ ]:
def _show_register() -> None:
    reg = risk_register(COMPONENTS, FLOWS, tainted_components(COMPONENTS, FLOWS), RULES)
    for asi, items in reg.items():
        print(f"  {asi} {ASI_TITLES[asi]:38s} {', '.join(items) if items else '-'}")
    hit = sum(bool(v) for v in reg.values())
    print(f"{hit} of {len(ASI_IDS)} categories have at least one exposed item, {sum(len(v) for v in reg.values())} exposures in all")
    chat = {"user": _c(3, "human"), "web": _c(0), "assistant": _c(2, "llm", "agent"), "fetch": _c(1, "tool", "reads_untrusted")}
    chat_flows = [_f("c1", "user", "assistant", "content"), _f("c2", "web", "fetch", "content"), _f("c3", "fetch", "assistant", "content")]
    chat_reg = risk_register(chat, chat_flows, tainted_components(chat, chat_flows), RULES)
    print(f"a chat assistant with one fetcher and nothing else is exposed to {sum(bool(v) for v in chat_reg.values())} of {len(ASI_IDS)} categories "
          f"({', '.join(a for a, v in chat_reg.items() if v) or 'none'}); the rules key on capabilities, not on the word 'agent'")
    near = risk_register(COMPONENTS, FLOWS, _one_hop(COMPONENTS, FLOWS), RULES)
    lost = sum(len(v) for v in reg.values()) - sum(len(v) for v in near.values())
    print(f"with the one-step view of taint from section 3, {lost} exposures drop out of the register")


_try("register", _show_register, needs=("exercise 2", "exercise 3"))

## 5. Exercise 4 — `mitigation_checklist`: from exposures to work items

A register says what is exposed. A **checklist** says what must be true about each exposed item. The
next cell is a small catalogue, this course's own, of mitigations for each category. One mitigation can
answer several categories (scoped credentials limit both a misused tool and an abused identity).

Write `mitigation_checklist(register, controls)`: one row `(item, mitigation, categories)` for every
distinct `(item, mitigation)` pair the register calls for, where `categories` is the sorted **tuple** of
every category that asks for it. Sort the rows by item, then mitigation. A category that has exposed items but
no entry in `controls` raises `ValueError` naming it: a risk with no listed mitigation must not
vanish from the list in silence. A key present with an empty list *is* an entry: it adds no rows and does not raise.

In [ ]:
CONTROLS = {
    "ASI01": ["untrusted-content-marked", "goal-lock-check"],
    "ASI02": ["tool-allowlist", "argument-validation", "human-approval-on-irreversible", "scoped-credentials"],
    "ASI03": ["scoped-credentials", "short-lived-tokens"],
    "ASI04": ["pinned-tool-versions", "tool-description-review"],
    "ASI05": ["sandboxed-execution", "human-approval-on-irreversible"],
    "ASI06": ["memory-write-validation", "memory-provenance-tags", "action-logging"],
    "ASI07": ["message-authentication"],
    "ASI08": ["step-and-spend-limit", "action-logging"],
    "ASI09": ["approval-shows-raw-source", "human-approval-on-irreversible"],
    "ASI10": ["kill-switch", "action-logging", "step-and-spend-limit"],
}

<details><summary>💡 Hint 1 — what to think about</summary>

Two categories can ask for the same mitigation on the same item. Should the checklist then carry one
row or two, and what should the row say about who asked? And what is the right behaviour when a category
has exposures and the catalogue has nothing for it?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Build a mapping keyed by `(item, mitigation)` whose value is the set of categories that asked. Walk the
register, raise for a category that has items and no entry, and add to the mapping. At the end sort the
keys and turn each set into a sorted tuple.

</details>

In [ ]:
def mitigation_checklist(register, controls):
    """The mitigations each exposed item needs, merged across categories.

    Args:
        register: dict category -> sorted list of item ids (exercise 3).
        controls: dict category -> list of mitigation ids.

    A category with exposed items and no key in ``controls`` raises ``ValueError``. A key with an empty
    list is an entry (no rows, no error).

    Returns:
        A list of ``(item, mitigation, categories)`` tuples, sorted by item then mitigation, one per distinct
        pair; ``categories`` is the sorted tuple of every category that asks for that pair.

    Example:
        >>> mitigation_checklist({"ASI02": ["t"], "ASI03": ["t"]}, {"ASI02": ["a", "b"], "ASI03": ["b"]})
        [('t', 'a', ('ASI02',)), ('t', 'b', ('ASI02', 'ASI03'))]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_mitigation_checklist() -> None:
    reg = {"ASI02": ["t", "u"], "ASI03": ["t"], "ASI05": []}
    ctl = {"ASI02": ["a", "b"], "ASI03": ["b", "c"]}
    got = mitigation_checklist(reg, ctl)
    want = [("t", "a", ("ASI02",)), ("t", "b", ("ASI02", "ASI03")), ("t", "c", ("ASI03",)), ("u", "a", ("ASI02",)), ("u", "b", ("ASI02",))]
    assert [tuple(r) for r in got] == want, (
        f"got {[tuple(r) for r in got]}. One row per distinct (item, mitigation), sorted by item then mitigation; "
        "when two categories ask for the same pair the row says both, as a sorted tuple.")
    try:
        mitigation_checklist({"ASI09": ["h"]}, {"ASI02": ["a"]})
    except ValueError:
        pass
    else:
        raise AssertionError("a category with exposed items and no mitigation listed must raise ValueError")
    print("exercise 4 looks right")


_try("exercise 4", _check_mitigation_checklist)

In [ ]:
def _show_checklist() -> None:
    reg = risk_register(COMPONENTS, FLOWS, tainted_components(COMPONENTS, FLOWS), RULES)
    rows = mitigation_checklist(reg, CONTROLS)
    naive = sum(len(items) * len(CONTROLS[asi]) for asi, items in reg.items())
    merged = [r for r in rows if len(r[2]) > 1]
    print(f"{naive} (item, category, mitigation) requests from the register become {len(rows)} checklist rows")
    print(f"{len(merged)} rows are asked for by more than one category: " + "; ".join(f"{i} / {m} ({', '.join(c)})" for i, m, c in merged))
    items = sorted({r[0] for r in rows})
    print(f"{len(items)} items carry work; the one with the most rows is {max(items, key=lambda i: sum(r[0] == i for r in rows))}")


_try("checklist", _show_checklist, needs=("exercise 2", "exercise 3", "exercise 4"))

## 6. Exercise 5 — `run_checks`: tests that fail when a mitigation is missing

A checklist nobody runs is a document. Each mitigation here has a **checker**, a function
`checker(arch, item) -> bool` that reads the item's configuration and says whether the mitigation is
in place. The next cell gives them: read how each decides. A checker here is deliberately fragile, and a
mitigation has no checker at all, because not everything can be tested yet.

Write `run_checks(checklist, arch, checkers)` that runs the checker of every row and returns the failing
rows as `(item, mitigation, reason)` in checklist order, with reason:

* `"missing"`: the checker returned anything other than the exact value `True`;
* `"error"`: the checker raised an `Exception`. A check that cannot run is not a pass: **fail closed**;
* `"no checker"`: the mitigation is not a key of `checkers`.

Catch `Exception`, not everything: the notebook's time limit must still be able to stop a checker.

In [ ]:
def _cfg(arch, item):
    return arch["config"].get(item, {})


CHECKERS = {
    "untrusted-content-marked": lambda arch, item: _cfg(arch, item).get("wraps_untrusted") is True,
    "tool-allowlist": lambda arch, item: bool(_cfg(arch, item).get("allowed_tools")) and "*" not in _cfg(arch, item)["allowed_tools"],
    "argument-validation": lambda arch, item: _cfg(arch, item).get("schema_validated") is True,
    "human-approval-on-irreversible": lambda arch, item: _cfg(arch, item).get("approval") == "required",
    "scoped-credentials": lambda arch, item: (bool(_cfg(arch, item).get("scopes"))
                                              and not any("*" in s for s in _cfg(arch, item)["scopes"])),
    "short-lived-tokens": lambda arch, item: _cfg(arch, item).get("token_ttl_s", 10 ** 9) <= 3600,
    "pinned-tool-versions": lambda arch, item: _cfg(arch, item).get("version") not in (None, "latest") and bool(_cfg(arch, item).get("digest")),
    "tool-description-review": lambda arch, item: _cfg(arch, item).get("description_reviewed") is True,
    "sandboxed-execution": lambda arch, item: _cfg(arch, item).get("sandbox") == "container" and _cfg(arch, item).get("network") == "none",
    "memory-write-validation": lambda arch, item: _cfg(arch, item).get("validate_writes") is True,
    "memory-provenance-tags": lambda arch, item: _cfg(arch, item).get("provenance") is True,
    "message-authentication": lambda arch, item: _cfg(arch, item)["auth"] in ("mtls", "signed"),        # KeyError when never set
    "step-and-spend-limit": lambda arch, item: _cfg(arch, item).get("max_steps", 10 ** 9) <= 25 and _cfg(arch, item).get("max_usd", 10 ** 9) <= 5,
    "approval-shows-raw-source": lambda arch, item: _cfg(arch, item).get("show_source") is True,
    "kill-switch": lambda arch, item: _cfg(arch, item).get("kill_switch") is True,
    "action-logging": lambda arch, item: _cfg(arch, item).get("log") == "append-only",
    # "goal-lock-check" has no checker: nobody has a reliable automatic test for it yet.
}

In [ ]:
def run_checks(checklist, arch, checkers):
    """Run every row's checker and return the rows that do not pass.

    Args:
        checklist: list of (item, mitigation, categories), as from exercise 4.
        arch: the architecture dict; passed to each checker unchanged.
        checkers: dict mitigation id -> function ``checker(arch, item)``.

    Returns:
        A list of ``(item, mitigation, reason)`` in checklist order for each row that failed: "no checker"
        when the mitigation has no entry in ``checkers``; "error" when the checker raised an ``Exception``;
        "missing" when it returned anything other than exactly ``True``. Passing rows are not listed.

    Example:
        >>> run_checks([("x", "m1", ()), ("x", "m2", ()), ("x", "m3", ())], {},
        ...            {"m1": lambda a, i: True, "m2": lambda a, i: 1 / 0})
        [('x', 'm2', 'error'), ('x', 'm3', 'no checker')]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_run_checks() -> None:
    rows = [("y", "m1", ()), ("x", "m2", ()), ("x", "m3", ()), ("y", "m4", ()), ("x", "m5", ()), ("x", "m6", ())]
    calls = []

    def seen(a, i):
        calls.append(i)
        return a == {"k": 1} and i in ("x", "y")
    ck = {"m1": seen, "m2": lambda a, i: {}["boom"], "m4": lambda a, i: "yes", "m5": lambda a, i: False, "m6": lambda a, i: 1}
    got = [tuple(r) for r in run_checks(rows, {"k": 1}, ck)]
    assert calls == ["y"], f"call checker(arch, item) exactly once per row, with the architecture you were given; got calls {calls}"
    assert got == [("x", "m2", "error"), ("x", "m3", "no checker"), ("y", "m4", "missing"), ("x", "m5", "missing"), ("x", "m6", "missing")], (
        f"got {got}. Only an exact True passes ('yes' is not True); a checker that raises is an 'error', not a pass and "
        "not a crash; a mitigation with no checker is 'no checker'; keep the checklist's order.")
    class _Halt(BaseException):
        """Stands in for the notebook's time limit, which is not an Exception either."""

    def halt(arch, item):
        raise _Halt()
    try:
        run_checks([("x", "h", ())], {}, {"h": halt})
    except _Halt:
        pass
    else:
        raise AssertionError("catch Exception, not BaseException: the time limit that protects this notebook must get through")
    print("exercise 5 looks right")


_try("exercise 5", _check_run_checks)

In [ ]:
HARDENING = [
    ("mark untrusted content", {"orchestrator": {"wraps_untrusted": True}}),
    ("scope and expire credentials", {"orchestrator": {"scopes": ["mail.send"], "token_ttl_s": 900},
                                      "mail_tool": {"scopes": ["mail.send"], "token_ttl_s": 900},
                                      "mcp_server": {"scopes": ["lookup.read"]}}),
    ("validate and approve tool calls", {"mail_tool": {"allowed_tools": ["send_mail"], "schema_validated": True, "approval": "required"},
                                         "mcp_server": {"allowed_tools": ["lookup"], "schema_validated": True, "approval": "required"},
                                         "code_tool": {"approval": "required"}, "approver": {"approval": "required"}}),
    ("isolate code and memory", {"code_tool": {"sandbox": "container", "network": "none"},
                                 "memory": {"validate_writes": True, "provenance": True, "log": "append-only"}}),
    ("pin and review the third-party server", {"mcp_server": {"version": "2.3.1", "digest": "sha256:0f3c", "description_reviewed": True}}),
    ("sign agent messages", {"f04": {"auth": "signed"}, "f05": {"auth": "signed"}}),
    ("limit, log and be able to stop", {"orchestrator": {"max_steps": 20, "max_usd": 2, "log": "append-only", "kill_switch": True},
                                        "approver": {"show_source": True}}),
]


CONTROLS_FLAT = sorted({m for ms in CONTROLS.values() for m in ms})


def _show_failures() -> None:
    reg = risk_register(COMPONENTS, FLOWS, tainted_components(COMPONENTS, FLOWS), RULES)
    rows = mitigation_checklist(reg, CONTROLS)
    failing = run_checks(rows, ARCH, CHECKERS)
    reasons = {r: sum(x[2] == r for x in failing) for r in ("missing", "error", "no checker")}
    print(f"{len(failing)} of {len(rows)} checks fail on the architecture as it is: "
          f"{reasons['missing']} missing, {reasons['error']} errored, {reasons['no checker']} without a checker")
    for item, mitigation, reason in failing[:6]:
        print(f"  FAIL {item:15s} {mitigation:32s} ({reason})")
    print(f"  ... and {max(0, len(failing) - 6)} more")
    lying = {m: (lambda arch, item: True) for m in CONTROLS_FLAT}
    print(f"with every checker replaced by one that answers True, the board shows {len(run_checks(rows, ARCH, lying))} failures: "
          "a green run proves only that the checkers agree, so the checkers need review as much as the code does")


_try("failing checks", _show_failures, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5"))

### Add the mitigations, one at a time

`HARDENING` above is a list of changes to the configuration, each a named step. The next cell applies
them in order to a **copy** and runs your checks after each, so you can see which step clears which
failures, and what is left at the end. Read the last lines: what remains is the part no configuration
change can clear.

In [ ]:
def _show_hardening() -> None:
    arch = copy.deepcopy(ARCH)
    reg = risk_register(arch["components"], arch["flows"], tainted_components(arch["components"], arch["flows"]), RULES)
    rows = mitigation_checklist(reg, CONTROLS)
    before = start_failing = len(run_checks(rows, arch, CHECKERS))
    print(f"{'start':42s} {before:3d} failing of {len(rows)}")
    for name, patch in HARDENING:
        for item, settings in patch.items():
            arch["config"].setdefault(item, {}).update(settings)
        now = len(run_checks(rows, arch, CHECKERS))
        print(f"{name:42s} {now:3d} failing   ({now - before:+d})")
        before = now
    left = run_checks(rows, arch, CHECKERS)
    print("left: " + ", ".join(f"{i} / {m} ({r})" for i, m, r in left))
    print(f"every remaining failure is a mitigation with no checker: {all(r == 'no checker' for _, _, r in left)}")
    print(f"the original architecture is unchanged by this cell (still {len(run_checks(rows, ARCH, CHECKERS))} failing): "
          f"{len(run_checks(rows, ARCH, CHECKERS)) == start_failing}")


_try("hardening", _show_hardening, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 7. Self-check

1. A teammate says the agent is safe because the only entry point for outside text is the web fetcher,
   and the fetcher is sandboxed. What does your taint result say about that?
   - (a) Nothing: the entry points are the only places where a threat model needs to look.
   - (b) Other components read and pass on the fetcher's output, so the exposure sits downstream.
   - (c) A sandbox on one component makes every component that it feeds trusted as well.
   - (d) Taint matters only for the components that are able to send mail outside.

2. Why is a flow that carries only an approval signal left out of the taint propagation?
   - (a) Approvals are always honest, so nothing needs to be tracked through them.
   - (b) A signal is encrypted in transit, so nothing can be injected into it.
   - (c) It carries a constrained value, not free text a receiver reads as instructions.
   - (d) It forms a cycle, and cycles are skipped by the propagation to avoid looping.

3. `run_checks` finds that a checker raised `KeyError`. What is the sound way to report the row?
   - (a) Passing, because nothing was found to be wrong with the configuration.
   - (b) Skipped, because a broken test says nothing either way about the item.
   - (c) Raise the `KeyError` to the caller so the whole run stops at that row.
   - (d) Failing as an error, since a check that could not run is no evidence of it.

4. After every mitigation with a checker is in place, some rows still fail. What does that tell you?
   - (a) Those mitigations cannot be shown by this suite, a gap to close and not to hide.
   - (b) The register has too many categories for the checklist to be completed.
   - (c) The configuration is wrong in some way that the checkers are unable to find.
   - (d) The checks should be dropped from the run so that the board reads all green.

Answers come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
if __name__ == "__main__":
    print("facts for the questions, printed here rather than taken on trust (they need your exercises to be done):")

    def _facts() -> None:
        downstream = [n for n in tainted_components(COMPONENTS, FLOWS) if n != "fetch_tool"]
        print(f"  question 1: besides the fetcher, {len(downstream)} components are reached by text that entered at the web or the inbox")
        quiet = [f["id"] for f in FLOWS if "content" not in f["tags"]]
        print(f"  question 2: the flows that carry no free text, which the propagation skips: {len(quiet)} ({', '.join(quiet)})")
        rows = mitigation_checklist(risk_register(COMPONENTS, FLOWS, tainted_components(COMPONENTS, FLOWS), RULES), CONTROLS)
        print(f"  questions 3 and 4: {sum(m not in CHECKERS for _, m, _ in rows)} of {len(rows)} rows name a mitigation with no checker")

    _try("facts for the self-check", _facts, needs=("exercise 2", "exercise 3", "exercise 4"))

## 8. Common mistakes

* **Taint from the entry points only.** Text passes through the research agent, the orchestrator and the
  memory; section 3 prints how many components a one-step view misses.
* **A tag rule that matches any tag instead of all of them.** It marks every tool as sending mail.
* **Dropping a category with no exposures from the register.** A reviewer reading the register needs to see
  the empty rows; absence of a key looks the same as never having asked.
* **Counting a checker that raises, or returns a truthy non-answer, as a pass.** A test that cannot run is not a pass.
* **Catching every exception, `BaseException` included.** The time limit that protects your notebook is
  an exception too; swallowing it hides a hung check.
* **Trusting a green board.** The checkers here read configuration; they do not attack anything. A passing
  run says the settings are present, not that the system resists a real injection (see the next lesson).

## 9. What this lesson does NOT cover

* **A real system.** The architecture, the rules, the catalogue and the settings are invented for teaching and
  simplified: a real threat model has finer trust levels, many more flows, and rules argued out with the people who
  run the system.
* **The OWASP document itself.** This lesson uses its identifiers and titles and nothing of its text beyond that.
  Read the source: the list is new, a first draft used different titles (for example "Agent Behaviour Hijack" for
  its first entry), and later editions may renumber or rename. As of 20 October 2026 the ten titles are those in the
  contents page of the published PDF (version 2026, December 2025). That document writes "&" in the title of ASI06
  but "and" in that of ASI03, and the lesson keeps its spelling. Titles of the list: OWASP GenAI Security Project,
  OWASP Top 10 for Agentic Applications 2026, CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/).
  Check them against the release you use.
* **Attacking the system.** The checks read configuration. Measuring how often a defence stops real injections
  needs attacks, a labelled set and a score; that is a different lesson.
* **Automatic discovery.** The architecture is typed in. Extracting it from a running system or from a tool
  server's manifest is a good next step and is not built here.

In [ ]:
def _recap() -> None:
    reg = risk_register(COMPONENTS, FLOWS, tainted_components(COMPONENTS, FLOWS), RULES)
    rows = mitigation_checklist(reg, CONTROLS)
    failing = run_checks(rows, ARCH, CHECKERS)
    print("--- what you built, measured ---")
    print(f"  {len(boundary_crossings(COMPONENTS, FLOWS))} boundary crossings; text from outside reaches {len(tainted_components(COMPONENTS, FLOWS))} of {len(COMPONENTS)} components")
    print(f"  {sum(len(v) for v in reg.values())} exposures across {sum(bool(v) for v in reg.values())} of {len(ASI_IDS)} categories -> {len(rows)} checklist rows -> {len(failing)} failing checks")
    print("  every number above was computed by the code you just ran")


if __name__ == "__main__":
    _try("the closing recap", _recap, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, function, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = f"   (waiting on `{waiting}`)" if status == "not started" and waiting not in (None, "", function) else ""
        print(f"  {_MARKS[status]}  {label} · {function}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, function, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]}, {next_up[1]}, in {next_up[2]}. Its cell has hints if you are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check that ran and
    came back wrong is. In a script or under CI that ends the run non-zero, so a green exit code can never
    paper over it. In a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.